# RL for Optimal Trade Execution

A DQN agent that makes per-second execution decisions based on the current order book state.
Instead of predicting 60 seconds ahead, the agent reacts to what it sees right now.

In [ ]:
import os, sys, subprocess

# Colab setup: clone repo + install deps
if os.path.exists("/content") and not os.path.isdir("/content/Ultramarin/data"):
    if not os.path.isdir("/content/Ultramarin/utils"):
        subprocess.run(["git", "clone", "-q", "-b", "potentially-improve-model",
                        "https://github.com/JosephZacharyGawlik/Ultramarin.git"],
                       cwd="/content")
    os.chdir("/content/Ultramarin")
    sys.path.insert(0, os.getcwd())

    # Mount Drive and copy data
    from google.colab import drive
    drive.mount('/content/drive')
    dst = "/content/Ultramarin/data"
    src = "/content/drive/MyDrive/data"
    os.makedirs(dst, exist_ok=True)
    subprocess.run(["cp", "-r", f"{src}/.", dst], capture_output=True, text=True)
    print("Data copied.")

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "gymnasium", "stable-baselines3"], capture_output=True)
print("Setup complete.")

In [ ]:
import numpy as np
import pandas as pd
import math
import warnings
import random
from pathlib import Path

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import DQN

from data.simulate_walk_the_book import simulate_walk_the_book

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
# === Change this to switch currencies ===
data_asset = "BTCUSDT"

KNOWN_VOLUMES = {
    "BTCUSDT": 4.0, "ETHUSDT": 55.0, "LTCUSDT": 51.0,
    "SOLUSDT": 315.0, "ADAUSDT": 10436.0, "DOGEUSDT": 60349.0, "XRPUSDT": 13736.0,
}
OPTIMAL_K = {
    "BTCUSDT": 14, "ETHUSDT": 26, "LTCUSDT": 16,
    "SOLUSDT": 17, "ADAUSDT": 7, "DOGEUSDT": 20, "XRPUSDT": 20,
}

volume_to_fill = KNOWN_VOLUMES[data_asset]
K_SECONDS = OPTIMAL_K[data_asset]

root = Path("data")
Y_path = root / data_asset / "y_train.parquet"

print(f"Currency: {data_asset}")
print(f"Volume to fill: {volume_to_fill}")
print(f"TWAP-K baseline: K={K_SECONDS}")

## Data Loading

Load Y data (the 60-second execution window), split into train/val using the same
chrono split as mhf-final (sort IDs, 80/20 split), and pre-extract LOB arrays per instrument.

In [ ]:
ASK_PRICE_COLS = [f'ask_price_{i}' for i in range(1, 6)]
ASK_VOL_COLS = [f'ask_vol_{i}' for i in range(1, 6)]
BID_PRICE_COLS = [f'bid_price_{i}' for i in range(1, 6)]
BID_VOL_COLS = [f'bid_vol_{i}' for i in range(1, 6)]

Y_raw = pd.read_parquet(Y_path).sort_values(["anonymized_id", "time_in_hour"])

# Get valid instrument IDs (exactly 60 rows)
id_counts = Y_raw.groupby("anonymized_id").size()
valid_ids = sorted(id_counts[id_counts == 60].index.tolist())

# Chrono split: same logic as train_val() — sort IDs, take last 20% as val
split_idx = int(len(valid_ids) * 0.8)
train_ids = valid_ids[:split_idx]
val_ids = valid_ids[split_idx:]

print(f"Valid instruments: {len(valid_ids)}")
print(f"Train: {len(train_ids)}, Val: {len(val_ids)}")


def extract_instruments(ids, df):
    """Pre-extract LOB arrays for each instrument."""
    instruments = []
    for aid in ids:
        df_inst = df[df["anonymized_id"] == aid].sort_values("time_in_hour")
        if len(df_inst) != 60:
            continue
        instruments.append({
            "id": aid,
            "ask_prices": df_inst[ASK_PRICE_COLS].to_numpy(dtype=np.float64),
            "ask_vols": df_inst[ASK_VOL_COLS].to_numpy(dtype=np.float64),
            "bid_prices": df_inst[BID_PRICE_COLS].to_numpy(dtype=np.float64),
            "bid_vols": df_inst[BID_VOL_COLS].to_numpy(dtype=np.float64),
            "close": df_inst["close"].dropna().iloc[-1],
        })
    return instruments


train_instruments = extract_instruments(train_ids, Y_raw)
val_instruments = extract_instruments(val_ids, Y_raw)

print(f"Train instruments loaded: {len(train_instruments)}")
print(f"Val instruments loaded: {len(val_instruments)}")

## Gymnasium Environment

Each episode = one instrument's 60-second execution window.  
State: 12 features (remaining vol/time, spread, depth, imbalance, momentum, etc.)  
Action: 6 discrete choices — multiples of the TWAP rate  
Reward: per-step IS relative to arrival price + terminal penalty for unfilled volume

In [ ]:
class TradeExecutionEnv(gym.Env):
    """
    RL environment for optimal trade execution.
    The agent decides how much to buy at each second of a 60-second window.
    """

    def __init__(self, instruments, volume_to_fill):
        super().__init__()
        self.instruments = instruments
        self.volume_to_fill = volume_to_fill
        self.twap_rate = volume_to_fill / 60

        # 6 discrete actions: multiples of TWAP rate
        self.action_space = spaces.Discrete(6)
        self.action_multipliers = np.array([0.0, 0.25, 0.5, 1.0, 2.0, 4.0])

        # 12 observation features
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(12,), dtype=np.float32
        )

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        # Pick instrument: specific index if provided, else random
        if options and "instrument_idx" in options:
            idx = options["instrument_idx"]
        else:
            idx = self.np_random.integers(len(self.instruments))
        self.inst = self.instruments[idx]

        # Reset execution state
        self.t = 0
        self.volume_executed = 0.0
        self.value_executed = 0.0
        self.volume_allocated = 0.0
        self.remaining_carry = 0.0

        # Arrival mid-price (benchmark for reward)
        self.arrival_mid = (
            self.inst["ask_prices"][0, 0] + self.inst["bid_prices"][0, 0]
        ) / 2

        return self._get_obs(), {}

    def _get_obs(self):
        t = self.t
        if t >= 60:
            return np.zeros(12, dtype=np.float32)

        ask_p = self.inst["ask_prices"][t]
        ask_v = self.inst["ask_vols"][t]
        bid_p = self.inst["bid_prices"][t]
        bid_v = self.inst["bid_vols"][t]
        mid = (ask_p[0] + bid_p[0]) / 2

        remaining_vol = max(0, self.volume_to_fill - self.volume_allocated)
        remaining_vol_frac = remaining_vol / (self.volume_to_fill + 1e-8)
        remaining_time_frac = (60 - t) / 60
        spread = (ask_p[0] - bid_p[0]) / (mid + 1e-8)
        mid_return = (mid - self.arrival_mid) / (self.arrival_mid + 1e-8)

        vol_imb_1 = (bid_v[0] - ask_v[0]) / (bid_v[0] + ask_v[0] + 1e-8)
        vol_imb_total = (bid_v.sum() - ask_v.sum()) / (bid_v.sum() + ask_v.sum() + 1e-8)
        ask_depth = ask_v.sum() / (self.volume_to_fill + 1e-8)
        bid_depth = bid_v.sum() / (self.volume_to_fill + 1e-8)

        cum_is = 0.0
        if self.volume_executed > 0:
            avg_price = self.value_executed / self.volume_executed
            cum_is = (avg_price - self.arrival_mid) / (self.arrival_mid + 1e-8)

        if t >= 5:
            mid_prev = (self.inst["ask_prices"][t-5, 0] + self.inst["bid_prices"][t-5, 0]) / 2
            momentum = (mid - mid_prev) / (self.arrival_mid + 1e-8)
            prev_spread = (self.inst["ask_prices"][t-5, 0] - self.inst["bid_prices"][t-5, 0]) / (mid_prev + 1e-8)
            spread_change = spread - prev_spread
        else:
            momentum = 0.0
            spread_change = 0.0

        carry_frac = self.remaining_carry / (self.volume_to_fill + 1e-8)

        return np.array([
            remaining_vol_frac,
            remaining_time_frac,
            spread,
            mid_return,
            vol_imb_1,
            vol_imb_total,
            ask_depth,
            bid_depth,
            cum_is,
            momentum,
            spread_change,
            carry_frac,
        ], dtype=np.float32)

    def step(self, action):
        position = self.action_multipliers[action] * self.twap_rate

        # Clip so cumulative doesn't exceed volume_to_fill
        max_remaining = self.volume_to_fill - self.volume_allocated
        position = min(position, max(0.0, max_remaining))

        # On last second, allocate all remaining volume
        if self.t == 59:
            position = max(0.0, max_remaining)

        self.volume_allocated += position

        # Walk the book for this second
        to_fill = position + self.remaining_carry
        self.remaining_carry = 0.0
        step_volume = 0.0
        step_cost = 0.0

        if to_fill > 1e-12:
            remaining = to_fill
            for level in range(5):
                price = self.inst["ask_prices"][self.t, level]
                avail = self.inst["ask_vols"][self.t, level]
                if math.isnan(price) or math.isnan(avail):
                    continue
                take = min(remaining, avail)
                step_cost += take * price
                step_volume += take
                remaining -= take
                if remaining <= 1e-12:
                    break
            self.remaining_carry = remaining

        self.volume_executed += step_volume
        self.value_executed += step_cost

        # Per-step reward: positive = bought below arrival (good for buyer)
        reward = 0.0
        if step_volume > 0:
            step_avg = step_cost / step_volume
            reward = (self.arrival_mid - step_avg) / self.arrival_mid
            reward *= step_volume / self.volume_to_fill  # scale by fraction filled

        self.t += 1
        done = self.t >= 60
        info = {}

        if done:
            # Terminal penalty for unfilled volume
            fill_ratio = self.volume_executed / (self.volume_to_fill + 1e-8)
            if fill_ratio < 0.99:
                reward -= (1.0 - fill_ratio) * 10.0

            # Log BPS for monitoring
            if self.volume_executed > 0:
                avg_price = self.value_executed / self.volume_executed
                close = self.inst["close"]
                bps = abs(avg_price - close) / close * 10000
                vol_penalty = min(100.0, self.volume_to_fill / self.volume_executed)
                info["bps"] = bps * vol_penalty
                info["fill_ratio"] = fill_ratio
            else:
                info["bps"] = 10000.0
                info["fill_ratio"] = 0.0

        obs = self._get_obs()
        return obs, reward, done, False, info


# Sanity check: run one episode with random actions
test_env = TradeExecutionEnv(train_instruments, volume_to_fill)
obs, _ = test_env.reset()
total_reward = 0
for _ in range(60):
    action = test_env.action_space.sample()
    obs, reward, done, _, info = test_env.step(action)
    total_reward += reward
    if done:
        break
print(f"Sanity check — Random agent: BPS={info.get('bps', 'N/A'):.4f}, "
      f"Fill={info.get('fill_ratio', 0):.2%}, Total reward={total_reward:.6f}")

## Training

DQN with a small MLP policy (2 hidden layers, 64 neurons each).  
200K timesteps = ~3,300 episodes = ~6 passes through the training data.

In [ ]:
train_env = TradeExecutionEnv(train_instruments, volume_to_fill)

model = DQN(
    "MlpPolicy",
    train_env,
    learning_rate=1e-3,
    buffer_size=50_000,
    learning_starts=1000,
    batch_size=64,
    tau=0.005,
    gamma=0.99,
    target_update_interval=500,
    exploration_fraction=0.3,
    exploration_initial_eps=1.0,
    exploration_final_eps=0.05,
    policy_kwargs={"net_arch": [64, 64]},
    verbose=1,
    seed=SEED,
)

TOTAL_TIMESTEPS = 200_000
print(f"Training for {TOTAL_TIMESTEPS} timesteps (~{TOTAL_TIMESTEPS // 60} episodes)...")
model.learn(total_timesteps=TOTAL_TIMESTEPS)
print("Training complete.")

## Evaluation

Run the trained agent on validation instruments and compare BPS against TWAP-K.  
Uses `simulate_walk_the_book()` for final BPS — same function as all other strategies.

In [ ]:
def evaluate(model, instruments, volume_to_fill, K_seconds):
    """Evaluate RL agent vs TWAP-K on a set of instruments."""
    env = TradeExecutionEnv(instruments, volume_to_fill)
    rl_bps_list = []
    twap_bps_list = []
    rl_positions_all = []

    for i in range(len(instruments)):
        inst = instruments[i]
        ask_prices = inst["ask_prices"]
        ask_vols = inst["ask_vols"]
        bid_prices = inst["bid_prices"]
        bid_vols = inst["bid_vols"]
        close_price = inst["close"]

        # --- RL Agent ---
        obs, _ = env.reset(options={"instrument_idx": i})
        rl_positions = np.zeros(60)
        for t in range(60):
            action, _ = model.predict(obs, deterministic=True)

            # Replicate the position logic from the env
            position = env.action_multipliers[int(action)] * env.twap_rate
            max_remaining = volume_to_fill - env.volume_allocated
            position = min(position, max(0.0, max_remaining))
            if t == 59:
                position = max(0.0, max_remaining)
            rl_positions[t] = position

            obs, _, done, _, _ = env.step(int(action))
            if done:
                break

        rl_positions_all.append(rl_positions)

        # Compute BPS via standard walk-the-book
        rl_vol, rl_avg = simulate_walk_the_book(
            rl_positions, ask_prices, ask_vols, bid_prices, bid_vols
        )
        if rl_vol > 0 and not np.isnan(rl_avg):
            ie = abs(rl_avg - close_price) / close_price * 10000
            vp = min(100.0, volume_to_fill / rl_vol)
            rl_bps_list.append(ie * vp)

        # --- TWAP-K ---
        twap_positions = np.zeros(60)
        twap_positions[-K_seconds:] = volume_to_fill / K_seconds

        twap_vol, twap_avg = simulate_walk_the_book(
            twap_positions, ask_prices, ask_vols, bid_prices, bid_vols
        )
        if twap_vol > 0 and not np.isnan(twap_avg):
            ie = abs(twap_avg - close_price) / close_price * 10000
            vp = min(100.0, volume_to_fill / twap_vol)
            twap_bps_list.append(ie * vp)

    return np.array(rl_bps_list), np.array(twap_bps_list), rl_positions_all


rl_bps, twap_bps, rl_positions_all = evaluate(
    model, val_instruments, volume_to_fill, K_SECONDS
)

print(f"\n{'='*55}")
print(f"RESULTS: {data_asset} (volume={volume_to_fill})")
print(f"{'='*55}")
print(f"{'Strategy':<20} {'Mean BPS':>10} {'Median BPS':>12}")
print(f"{'-'*55}")
print(f"{'RL Agent':<20} {rl_bps.mean():>10.4f} {np.median(rl_bps):>12.4f}")
print(f"{'TWAP-' + str(K_SECONDS) + 's':<20} {twap_bps.mean():>10.4f} {np.median(twap_bps):>12.4f}")
print(f"{'-'*55}")
diff = twap_bps.mean() - rl_bps.mean()
print(f"RL vs TWAP: {diff:+.4f} bps ({'RL wins' if diff > 0 else 'TWAP wins'})")
print(f"Instruments evaluated: {len(rl_bps)}")
print(f"RL wins on: {(rl_bps < twap_bps).sum()}/{len(rl_bps)} instruments "
      f"({(rl_bps < twap_bps).mean()*100:.1f}%)")
print(f"{'='*55}")

## Policy Analysis

Examine what the agent learned: average position profile across instruments,
compared to TWAP allocation.

In [ ]:
# Average position profile
positions_array = np.array(rl_positions_all)  # [num_instruments, 60]
avg_positions = positions_array.mean(axis=0)

# TWAP-K profile for comparison
twap_profile = np.zeros(60)
twap_profile[-K_SECONDS:] = volume_to_fill / K_SECONDS

# TWAP-60 profile
twap60_profile = np.full(60, volume_to_fill / 60)

print("RL Agent — Average position profile:")
print(f"  Volume in first 30s:  {avg_positions[:30].sum():.4f} ({avg_positions[:30].sum()/volume_to_fill*100:.1f}%)")
print(f"  Volume in last 30s:   {avg_positions[30:].sum():.4f} ({avg_positions[30:].sum()/volume_to_fill*100:.1f}%)")
print(f"  Volume in last {K_SECONDS}s:   {avg_positions[-K_SECONDS:].sum():.4f} ({avg_positions[-K_SECONDS:].sum()/volume_to_fill*100:.1f}%)")
print(f"  Volume in last 5s:    {avg_positions[-5:].sum():.4f} ({avg_positions[-5:].sum()/volume_to_fill*100:.1f}%)")
print()
print(f"Top 5 seconds by average position:")
top5 = np.argsort(avg_positions)[-5:][::-1]
for s in top5:
    print(f"  Second {s}: {avg_positions[s]:.4f} ({avg_positions[s]/volume_to_fill*100:.1f}%)")

## Generate Test Positions

Run the agent on test data and save positions in the submission format.

In [ ]:
# Load test data
X_test_path = root / data_asset / "X_test.parquet"
if X_test_path.exists():
    # Test data doesn't have Y, so we need to check if y_test exists
    # For now, generate positions using validation instruments as a demo
    print("Test position generation: use val instruments as demo")
    print("(Replace with actual test data when available)")

    times = Y_raw["time_in_hour"].sort_values().unique()

    test_positions_df = pd.DataFrame()
    env = TradeExecutionEnv(val_instruments, volume_to_fill)

    for i, inst in enumerate(val_instruments):
        obs, _ = env.reset(options={"instrument_idx": i})
        positions = np.zeros(60)
        for t in range(60):
            action, _ = model.predict(obs, deterministic=True)
            pos = env.action_multipliers[int(action)] * env.twap_rate
            max_rem = volume_to_fill - env.volume_allocated
            pos = min(pos, max(0.0, max_rem))
            if t == 59:
                pos = max(0.0, max_rem)
            positions[t] = pos
            obs, _, done, _, _ = env.step(int(action))
            if done:
                break

        df = pd.DataFrame({
            'anonymized_id': inst['id'],
            'time_in_hour': times,
            'position': positions,
        })
        test_positions_df = pd.concat([test_positions_df, df], ignore_index=True)

    base_path = Path(f"positions/{data_asset}")
    base_path.mkdir(parents=True, exist_ok=True)
    target = base_path / "rl_agent.parquet"
    test_positions_df.to_parquet(target, index=False, engine='pyarrow')
    print(f"Saved {len(test_positions_df)} rows to {target}")
else:
    print(f"Test data not found at {X_test_path}")